<a href="https://colab.research.google.com/github/GuByeongMo99/4-data_science/blob/main/3%EC%A3%BC%EC%B0%A8/3_Apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Apriori & FP tree

In [ ]:
!pip install pip --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [ ]:
!pip install mlxtend==0.18

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 10.7 MB/s  0:00:00
  Attempting uninstall: mlxtend
    Found existing installation: mlxtend 0.23.4
    Uninstalling mlxtend-0.23.4:
      Successfully uninstalled mlxtend-0.23.4


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import mlxtend
mlxtend.__version__

'0.18.0'

In [5]:
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

In [6]:
import pandas as pd
import os

## Dataset import 및 확인

In [7]:
from google.colab import drive
drive.mount('/content/drive')

basicpath = '/content/drive/MyDrive/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
path = basicpath + '데이터_사이언스/3주차'
file = 'BreadBasket_DMS.csv'
data = pd.read_csv(os.path.join(path, file), index_col = None)

In [9]:
data.head(10)

,Date,Time,Transaction,Item
0,2016.10.30,9:58:11,1,Bread
1,2016.10.30,10:05:34,2,Scandinavian
2,2016.10.30,10:05:34,2,Scandinavian
3,2016.10.30,10:07:57,3,Hot chocolate
4,2016.10.30,10:07:57,3,Jam
5,2016.10.30,10:07:57,3,Cookies
6,2016.10.30,10:08:41,4,Muffin
7,2016.10.30,10:13:03,5,Coffee
8,2016.10.30,10:13:03,5,Pastry
9,2016.10.30,10:13:03,5,Bread


In [10]:
data.shape

(21293, 4)

## 과제1-1 Data Cleaning


위 데이터를 가공하여, transaction과 item이 column이 되는  
다음과 같은 형태의 새로운 DataFrame을 만들어 변수 data에 저장하시오.

 transaction | item
:-------------:|:------:
      0      |   7  
      0      |   14
      1      |   9  
      2      |   18
...  


In [11]:
# 1. 대문자 -> 소문자 변환
data['Item'] = data['Item'].str.lower()

In [12]:
# 2. 'none' 값 제거 및 필요한 컬럼만 선택
data = data[data['Item'] != 'none']

# 과제에서 요구한 transaction과 item 컬럼 형태 유지
data = data[['Transaction', 'Item']]
display(data.head())

,Transaction,Item
0,1,bread
1,2,scandinavian
2,2,scandinavian
3,3,hot chocolate
4,3,jam


### Data 간단 분석_item당 등장횟수

In [14]:
data

,Transaction,Item
0,1,bread
1,2,scandinavian
2,2,scandinavian
3,3,hot chocolate
4,3,jam
...,...,...
21288,9682,coffee
21289,9682,tea
21290,9683,coffee
21291,9683,pastry


### 과제 1-2 코드 설명
`encode_units` 함수와 `applymap`을 사용하는 이유는 다음과 같습니다:

1.  **데이터 이진화**: 연관 규칙 분석 알고리즘(Apriori, FP-Growth)은 특정 항목의 '구매 횟수'가 아니라 '구매 여부(0 또는 1)'를 입력값으로 받습니다.
2.  **중복 제거**: 한 번의 트랜잭션(영수증)에서 같은 아이템이 여러 번 등장하여 숫자가 1보다 커진 경우, 이를 모두 '구매함(1)' 상태로 변경하여 분석의 규격을 맞춥니다.

In [13]:
len(data['Item'].unique())

95

In [ ]:
top10_items = data['Item'].value_counts().head(10)
top10_items

,count
Item,
coffee,5471
bread,3325
tea,1435
cake,1025
pastry,856
sandwich,771
medialuna,616
hot chocolate,590
cookies,540


In [ ]:
len(data['Transaction'].unique())

9531

## 과제 1-2 one-hot-encoding vector 만들기
위 data를 가공하여 one-hot-encoding 형태의 데이터를 만들고  
이를 hot_encoded_data 변수에 저장하시오  
이때 one-hot-encoding 형태의 column은 item, row는 transaction이어야 함  

In [15]:
# 1. pd.crosstab을 사용하여 Transaction별 Item 출현 횟수 계산
hot_encoded_data = pd.crosstab(data['Transaction'], data['Item'])

display(hot_encoded_data.head())

Item,adjustment,afternoon with the baker,alfajores,argentina night,art tray,bacon,baguette,bakewell,bare popcorn,basket,...,the bart,the nomad,tiffin,toast,truffles,tshirt,valentine's card,vegan feast,vegan mincepie,victorian sponge
Transaction,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
def encode_units(x):
    # 1 이상의 값은 구매를 의미하므로 1로, 0 이하는 0으로 변환하여 이진화합니다.
    if x <= 0:
        return 0
    if x >= 1:
        return 1

# applymap을 통해 데이터프레임의 모든 요소에 encode_units 함수를 적용하여
# 연관 규칙 분석(Apriori)에 적합한 데이터 형태로 만듭니다.
hot_encoded_data = hot_encoded_data.applymap(encode_units)
display(hot_encoded_data.head())

/tmp/ipykernel_1414/3841275660.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  hot_encoded_data = hot_encoded_data.applymap(encode_units)


Item,adjustment,afternoon with the baker,alfajores,argentina night,art tray,bacon,baguette,bakewell,bare popcorn,basket,...,the bart,the nomad,tiffin,toast,truffles,tshirt,valentine's card,vegan feast,vegan mincepie,victorian sponge
Transaction,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 과제 1-3
mlxtend의 apriori & fp-growth 라이브러리를 활용하여 위 데이터에서 frequent itemset을 추출하시오.  
이때, min_support는 0.02로 하시오.  
association_rules 라이브러리를 활용하여 추출한 frequent itemset에서 association rule을 만드시오.

In [17]:
# 1. apriori를 활용한 빈번 항목 집합 추출
frequent_itemsets = apriori(hot_encoded_data, min_support=0.02, use_colnames=True)

# 2. 연관 규칙 생성
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1)
display(rules.head())

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction
0,(bread),(pastry),0.327205,0.086107,0.029160,0.089119,1.034977,0.000985,1.003306
1,(pastry),(bread),0.086107,0.327205,0.029160,0.338650,1.034977,0.000985,1.017305
2,(coffee),(cake),0.478394,0.103856,0.054728,0.114399,1.101515,0.005044,1.011905
3,(cake),(coffee),0.103856,0.478394,0.054728,0.526958,1.101515,0.005044,1.102664
4,(cake),(tea),0.103856,0.142631,0.023772,0.228891,1.604781,0.008959,1.111865


In [18]:
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1)

In [19]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction
0,(bread),(pastry),0.327205,0.086107,0.029160,0.089119,1.034977,0.000985,1.003306
1,(pastry),(bread),0.086107,0.327205,0.029160,0.338650,1.034977,0.000985,1.017305
2,(coffee),(cake),0.478394,0.103856,0.054728,0.114399,1.101515,0.005044,1.011905
3,(cake),(coffee),0.103856,0.478394,0.054728,0.526958,1.101515,0.005044,1.102664
4,(cake),(tea),0.103856,0.142631,0.023772,0.228891,1.604781,0.008959,1.111865
5,(tea),(cake),0.142631,0.103856,0.023772,0.166667,1.604781,0.008959,1.075372
6,(coffee),(cookies),0.478394,0.054411,0.028209,0.058966,1.083723,0.002179,1.004841
7,(cookies),(coffee),0.054411,0.478394,0.028209,0.518447,1.083723,0.002179,1.083174
8,(hot chocolate),(coffee),0.058320,0.478394,0.029583,0.507246,1.060311,0.001683,1.058553
9,(coffee),(hot chocolate),0.478394,0.058320,0.029583,0.061837,1.060311,0.001683,1.003749


In [ ]:
# 1. fp-growth를 활용한 빈번 항목 집합 추출
frequent_itemsets = fpgrowth(hot_encoded_data, min_support=0.02, use_colnames=True)

# 2. 연관 규칙 생성
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1)
display(rules.head())

/usr/local/lib/python3.11/dist-packages/mlxtend/frequent_patterns/fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [ ]:
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1)

In [ ]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(hot chocolate),(coffee),0.058320,0.478394,0.029583,0.507246,1.060311,1.0,0.001683,1.058553,0.060403,0.058333,0.055314,0.284542
1,(coffee),(hot chocolate),0.478394,0.058320,0.029583,0.061837,1.060311,1.0,0.001683,1.003749,0.109048,0.058333,0.003735,0.284542
2,(cake),(hot chocolate),0.103856,0.058320,0.011410,0.109868,1.883874,1.0,0.005354,1.057910,0.523553,0.075683,0.054740,0.152760
3,(hot chocolate),(cake),0.058320,0.103856,0.011410,0.195652,1.883874,1.0,0.005354,1.114125,0.498236,0.075683,0.102434,0.152760
4,(cookies),(coffee),0.054411,0.478394,0.028209,0.518447,1.083723,1.0,0.002179,1.083174,0.081700,0.055905,0.076787,0.288707
5,(coffee),(cookies),0.478394,0.054411,0.028209,0.058966,1.083723,1.0,0.002179,1.004841,0.148110,0.055905,0.004818,0.288707
6,(muffin),(coffee),0.038457,0.478394,0.018806,0.489011,1.022193,1.0,0.000408,1.020777,0.022579,0.037760,0.020354,0.264161
7,(coffee),(muffin),0.478394,0.038457,0.018806,0.039311,1.022193,1.0,0.000408,1.000888,0.041623,0.037760,0.000888,0.264161
8,(pastry),(coffee),0.086107,0.478394,0.047544,0.552147,1.154168,1.0,0.006351,1.164682,0.146161,0.091968,0.141396,0.325764
9,(coffee),(pastry),0.478394,0.086107,0.047544,0.099382,1.154168,1.0,0.006351,1.014740,0.256084,0.091968,0.014526,0.325764
